# From GCSE quality to A-level, with institution type included

This notebook repeats `a-level-school-quality.ipynb` with **institution type added**, so that each result can be compared with the original. It keeps everything else the same: the 1,883 schools with GCSE results, the seven subject groups (Maths, Sciences, English, Humanities, Social sciences, Business & Computing, Creative arts), general GCSE quality with regional means, the Maths-and-Science versus English-and-Open tilt, consistency, and the two ways of linking to A-level: a **flexible** model (each group has its own slopes on the two GCSE dimensions) and a model with **one A-level quality** tied to GCSE.

**What type adds.** For the four state types that make up almost all of these schools (academy converter, academy sponsor-led, free school / UTC / studio, LA maintained; from the public school register), a shift in general GCSE quality, in the tilt and in consistency, and an A-level shift beyond the GCSE profile: for each subject group in the flexible model, and for the single A-level quality in the other. The 19 colleges and other institutions that have GCSE results get a separate A-level offset. Independent schools have no GCSE results, so they are not in this sample (they are in `a-level-institution-type.ipynb`).

**What is deliberately not changed.** Region enters as before (a regional mean of general GCSE quality and a regional shift in A-level quality) and the local-authority layer used in the later notebooks is left out, so that the effect of adding type is not mixed with the effect of adding geography.

**The original results** (flexible model without type) are stored in a cell below and shown next to the new ones. The fits use four chains of 500 draws after 1,500 tuning steps (the original used 1,000 after 2,000), because the notebook has to fit in the machine's memory; each fit's results are extracted and the raw fit is released before the next one.

**This is association, not effect.** The GCSE quantities describe this year's Year 11, not the A-level students' cohort, and type differences reflect who was chosen to convert and where sponsor-led academies were created, not the effect of the type itself.

In [ ]:
import arviz as az
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import pytensor.tensor as pt
import xarray as xr
import gc

%config InlineBackend.figure_format = 'retina'
RANDOM_SEED = 8927
rng = np.random.default_rng(RANDOM_SEED)
az.style.use("arviz-darkgrid")
print(f"Running on PyMC v{pm.__version__}")

## Data

In [ ]:
raw = pd.read_csv("data/all-value-add-errors.csv")

elements = ["English", "Maths", "Science", "Humanities", "Languages", "Open"]
column = {"English": "P8MEAENG", "Maths": "P8MEAMAT", "Science": "SCIVAMEA_PTQ_EE",
          "Humanities": "HUMVAMEA_PTQ_EE", "Languages": "LANVAMEA_PTQ_EE", "Open": "P8MEAOPEN"}
frames = []
for e in elements:
    c = column[e]
    sub = raw[["URN", c, f"{c} lower", f"{c} upper"]].dropna()
    sub.columns = ["URN", "va", "lower", "upper"]
    sub["element"] = e
    frames.append(sub)
long = pd.concat(frames)
long["se"] = (long["upper"] - long["lower"]) / (2 * 1.96)
n_elements_per_school = long.groupby("URN")["element"].nunique()
long = long[long["URN"].isin(n_elements_per_school[n_elements_per_school >= 3].index)].sort_values("URN").reset_index(drop=True)

urns = pd.Index(sorted(long["URN"].unique()))
long["school_idx"] = urns.get_indexer(long["URN"])
long["element_idx"] = long["element"].map({e: k for k, e in enumerate(elements)}).to_numpy()
n_schools, n_elements = len(urns), len(elements)
x_obs, x_se = long["va"].to_numpy(), long["se"].to_numpy()
s_idx, e_idx = long["school_idx"].to_numpy(), long["element_idx"].to_numpy()
counts = np.bincount(s_idx, minlength=n_schools)

region_series = raw.drop_duplicates("URN").set_index("URN")["RGN24NM"].reindex(urns)
regions = list(region_series.value_counts().index)
reg_idx = region_series.map({r: k for k, r in enumerate(regions)}).fillna(len(regions)).astype(int).to_numpy()   # unknown region -> last index (national average)
print(f"{n_schools} schools, {len(long)} GCSE school-element observations; {(reg_idx == len(regions)).sum()} schools without a region")


# ---- institution type, from the public register's detailed establishment type
names = pd.read_csv("data/school-names.csv").set_index("URN")
def type_category(t):
    if t in ("Other independent school", "Other independent special school"):
        return "independent (fee-paying)"
    if t in ("Academy converter", "Academy special converter"):
        return "academy converter"
    if t in ("Academy sponsor led", "Academy special sponsor led"):
        return "academy sponsor-led"
    if t in ("Free schools", "Free schools special", "University technical college", "Studio schools", "City technology college", "Free schools alternative provision"):
        return "free school / UTC / studio"
    if t in ("Community school", "Voluntary aided school", "Voluntary controlled school", "Foundation school"):
        return "LA maintained"
    if t in ("Further education", "Sixth form centres", "Academy 16-19 converter", "Academy 16 to 19 sponsor led", "Free schools 16 to 19"):
        return "college (post-16)"
    return "other"
type_names = ["independent (fee-paying)", "academy converter", "academy sponsor-led", "free school / UTC / studio", "LA maintained", "college (post-16)", "other"]
state4 = type_names[1:5]
type_o_names = ["college (post-16)", "other"]
school_type = np.array([type_names.index(type_category(t)) for t in names["type_detail"].reindex(urns).to_numpy()])
is4 = ((school_type >= 1) & (school_type <= 4)).astype(float)       # the four state types
t4_idx = np.where(is4 == 1, school_type - 1, 0)
is_o = 1.0 - is4                                                   # the few colleges and other institutions that have GCSE results
o_idx = np.where(school_type == 5, 0, 1)
print(pd.Series(np.array(type_names)[school_type]).value_counts().reindex(type_names).fillna(0).astype(int).to_string())

In [ ]:
arts = ["Art & Design", "Art & Design (Fine Art)", "Art & Design (Photography)", "Art & Design (Graphics)", "Art & Design (Textiles)",
        "Art & Design (3d Studies)", "Art & Design (Critical Studies)", "Music", "Music Technology", "Drama & Theatre Studies", "Dance"]
groups = {"Maths": ["Mathematics"],
          "Sciences": ["Biology", "Chemistry", "Physics"],
          "English": ["English Literature", "English Language", "English Language & Literature"],
          "Humanities": ["History", "Geography", "Religious Studies", "Logic/ Philosophy", "Ancient History", "Classical Civilisation"],
          "Social sciences": ["Psychology", "Sociology", "Economics", "Government & Politics", "Law"],
          "Business & Computing": ["Business Studies:Single", "Computer Studies/Computing"],
          "Creative arts": arts}
group_names = list(groups)
n_groups = len(group_names)

raw_g = raw.set_index("URN").reindex(urns)
frames = []
for gname, subjects in groups.items():
    va = pd.DataFrame({s: raw_g[f"A-level {s} VA"] for s in subjects})
    se = pd.DataFrame({s: (raw_g[f"A-level {s} VA upper"] - raw_g[f"A-level {s} VA lower"]) / (2 * 1.96) for s in subjects})
    ent = pd.DataFrame({s: raw_g[f"A-level {s} entries"].where(va[s].notna(), 0).fillna(0) for s in subjects})
    total = ent.sum(axis=1)
    pooled_va = (va.fillna(0) * ent).sum(axis=1) / total.replace(0, np.nan)
    pooled_se = np.sqrt(((se.fillna(0) * ent) ** 2).sum(axis=1)) / total.replace(0, np.nan)   # entry-weighted; assumes separate cohorts
    f = pd.DataFrame({"school_idx": np.arange(n_schools), "group": gname, "va": pooled_va.to_numpy(), "se": pooled_se.to_numpy(), "entries": total.to_numpy()})
    frames.append(f.dropna(subset=["va", "se"]))
alevel = pd.concat(frames).reset_index(drop=True)
alevel["group_idx"] = alevel["group"].map({g: k for k, g in enumerate(group_names)}).to_numpy()
y_obs, y_se = alevel["va"].to_numpy(), alevel["se"].to_numpy()
ys_idx, yg_idx = alevel["school_idx"].to_numpy(), alevel["group_idx"].to_numpy()
groups_per_school = np.bincount(ys_idx, minlength=n_schools)
print(f"{len(alevel)} school-group A-level observations; groups per school: {({int(k): int(v) for k, v in zip(*np.unique(groups_per_school, return_counts=True))})}")
assert (y_se > 0).all()

### The original results, for comparison

Values from `a-level-school-quality.ipynb` (flexible model, no type, same data and groups). The region values are the regional shifts in general GCSE quality ($m_r$, SD units) and in the A-level quality beyond GCSE ($\psi_r$).

In [ ]:
orig = {"gcse_explains": [0.279, 0.275, 0.172, 0.191, 0.119, 0.077, 0.042],
        "general": [0.071, 0.078, 0.142, 0.177, 0.104, 0.071, 0.036], "tilt": [0.207, 0.197, 0.029, 0.014, 0.015, 0.005, 0.006],
        "b": [0.100, 0.088, 0.098, 0.105, 0.094, 0.078, 0.079], "c": [0.178, 0.146, -0.045, -0.029, -0.035, 0.017, -0.026],
        "true_sd": [0.391, 0.328, 0.271, 0.260, 0.303, 0.307, 0.441],
        "m": {"London": 0.615, "South East": 0.142, "South West": 0.067, "East of England": 0.047, "Yorkshire and The Humber": -0.096,
              "West Midlands": -0.108, "East Midlands": -0.124, "North West": -0.194, "North East": -0.350},
        "psi": {"London": 0.072, "South East": -0.042, "South West": -0.012, "East of England": 0.163, "Yorkshire and The Humber": 0.057,
                "West Midlands": -0.135, "East Midlands": 0.014, "North West": 0.030, "North East": -0.148},
        "one_quality": {"beta_g": 0.126, "beta_h": 0.057, "sigma_u": 0.284, "corr with general quality": 0.416, "corr with tilt": 0.181, "share explained by g and tilt": 0.199}}
pd.DataFrame({"GCSE explains (g + tilt)": orig["gcse_explains"], "b": orig["b"], "c": orig["c"], "true SD": orig["true_sd"]}, index=group_names)

## Models

In [ ]:
keep = np.array([0.0 if e == "Humanities" else 1.0 for e in elements])
is_maths_group = np.array([g == "Maths" for g in group_names])

def build_model(kind):
    """kind = 'flex': each A-level group has its own slopes on general quality g and tilt h, and its own type shifts.
       kind = 'latent': one A-level quality a_i = beta_g g_i + beta_h h_i + region + type + u_i, and the groups load on it."""
    with pm.Model(coords={"element": elements, "region": regions, "group": group_names, "type4": state4, "type_o": type_o_names}) as model:
        # ---- GCSE side: where general quality, the tilt and consistency sit, by region and by type
        mu = pm.Normal("mu", 0, 1, dims="element")
        lam = pm.HalfNormal("lam", 1, dims="element")
        tau = pm.HalfNormal("tau", 0.5, dims="element")
        sigma_m = pm.HalfNormal("sigma_m", 0.5)
        m = pm.ZeroSumNormal("m", sigma=sigma_m, dims="region")
        m_all = pt.concatenate([m, pt.zeros(1)])
        type_g = pm.ZeroSumNormal("type_g", sigma=0.5, dims="type4")       # type shift in general GCSE quality
        type_h = pm.ZeroSumNormal("type_h", sigma=0.5, dims="type4")       # type shift in the tilt
        type_c = pm.ZeroSumNormal("type_c", sigma=0.3, dims="type4")       # type shift in log consistency
        g_loc = m_all[reg_idx] + is4 * type_g[t4_idx]
        g = pm.Normal("g", g_loc, 1, shape=n_schools)
        h = pm.Normal("h", is4 * type_h[t4_idx], 1, shape=n_schools)
        k_raw = pm.Normal("kappa_raw", 0, 0.5, shape=n_elements)
        kappa = pm.Deterministic("kappa", k_raw * keep, dims="element")   # Humanities fixed at 0; the sign of the tilt is fixed after sampling
        sigma_s = pm.HalfNormal("sigma_s", 0.5)
        rho = pm.Deterministic("rho", 2 * pm.Beta("rho_raw", 2, 2) - 1)
        w = pm.Normal("w", 0, 1, shape=n_schools)
        log_s = is4 * type_c[t4_idx] + sigma_s * (rho * (g - g_loc) + pt.sqrt(1 - rho**2) * w)      # not stored: unused below
        pm.Normal("x_obs", mu=mu[e_idx] + lam[e_idx] * g[s_idx] + kappa[e_idx] * h[s_idx],
                  sigma=pt.sqrt((tau[e_idx] * pt.exp(log_s[s_idx]))**2 + x_se**2), observed=x_obs)

        # ---- A-level side
        nu = pm.Normal("nu", 0, 0.5, dims="group")
        sd_group = pm.HalfNormal("sd_group", 0.5, dims="group")
        sigma_psi = pm.HalfNormal("sigma_psi", 0.3)
        psi = pm.ZeroSumNormal("psi", sigma=sigma_psi, dims="region")
        psi_all = pt.concatenate([psi, pt.zeros(1)])
        u = pm.Normal("u", 0, 1, shape=n_schools)
        d_o = pm.Normal("d_o", 0, 0.5, dims=("group", "type_o"))             # mean shift for the colleges and other institutions that have GCSE results
        offset = is_o[ys_idx] * d_o[yg_idx, o_idx[ys_idx]]
        if kind == "flex":
            b = pm.Normal("b", 0, 0.5, dims="group")
            c = pm.Normal("c", 0, 0.5, dims="group")
            lam_a = pm.HalfNormal("lam_a", 0.5, dims="group")
            type_a = pm.ZeroSumNormal("type_a", sigma=0.3, dims=("group", "type4"))   # type shift at A-level beyond GCSE, by subject group
            shared = psi_all[reg_idx] + u
            y_mean = (nu[yg_idx] + b[yg_idx] * g[ys_idx] + c[yg_idx] * h[ys_idx] + lam_a[yg_idx] * shared[ys_idx]
                      + is4[ys_idx] * type_a[yg_idx, t4_idx[ys_idx]] + offset)
        else:
            beta_g = pm.Normal("beta_g", 0, 0.5)
            beta_h = pm.Normal("beta_h", 0, 0.5)
            sigma_u = pm.HalfNormal("sigma_u", 0.5)
            type_q = pm.ZeroSumNormal("type_q", sigma=0.3, dims="type4")              # type shift in the one A-level quality, beyond GCSE
            lam_raw = pm.HalfNormal("lam_raw", 1, shape=n_groups)
            lam_a = pm.Deterministic("lam_a", pt.where(is_maths_group, 1.0, lam_raw), dims="group")   # Maths sets the scale of A-level quality
            quality = beta_g * g + beta_h * h + psi_all[reg_idx] + is4 * type_q[t4_idx] + sigma_u * u
            y_mean = nu[yg_idx] + lam_a[yg_idx] * quality[ys_idx] + offset
        pm.Normal("y_obs", mu=y_mean, sigma=pt.sqrt(sd_group[yg_idx]**2 + y_se**2), observed=y_obs)
    return model

### Fit the flexible model

Each fit is followed by a helper that fixes the sign of the tilt (it is defined only up to sign, so each chain is flipped to the orientation with Maths and Science positive and English and Open negative), keeps the results the analysis needs and releases the raw fit.

In [ ]:
def align_and_extract(idata, kind, thin=2):
    post = idata.posterior
    k = post["kappa"]
    score = (k.sel(element="Maths") + k.sel(element="Science") - k.sel(element="English") - k.sel(element="Open")).mean("draw")
    sign = xr.where(score > 0, 1.0, -1.0)
    for name in ["h", "kappa", "type_h"] + (["c"] if kind == "flex" else ["beta_h"]):
        post[name] = post[name] * sign
    n_div = int(idata.sample_stats["diverging"].sum())
    ds = post.to_dataset()
    per_school = ["g", "h", "u"]
    glob = [v for v in ds.data_vars if v not in per_school + ["w", "kappa_raw", "rho_raw"]]
    small = xr.Dataset({**{v: ds[v] for v in glob}, **{v: ds[v].isel({ds[v].dims[-1]: slice(0, None, 10)}) for v in per_school}})
    rh, es = az.rhat(small), az.ess(small)
    diag = pd.DataFrame({"max r_hat": {v: float(rh[v].max()) for v in rh.data_vars}, "min bulk ESS": {v: float(es[v].min()) for v in es.data_vars}}).sort_values("max r_hat", ascending=False)
    R = {}
    for v in glob + per_school:
        a = ds[v].to_numpy(); R[v] = a.reshape(-1, *a.shape[2:])[::thin]
    return R, diag, n_div, int((sign.to_numpy() < 0).sum())

flex_model = build_model("flex")
with flex_model:
    idata_f = pm.sample(draws=500, tune=1500, chains=4, target_accept=0.99, random_seed=RANDOM_SEED, progressbar=False)
R_f, diag_f, div_f, flipped_f = align_and_extract(idata_f, "flex")
del idata_f, flex_model; gc.collect()
print(f"flexible model: divergences = {div_f}, chains flipped = {flipped_f} of 4")
display(diag_f.round(3).head(8))

### Fit the one-A-level-quality model

In [ ]:
latent_model = build_model("latent")
with latent_model:
    idata_l = pm.sample(draws=500, tune=1500, chains=4, target_accept=0.99, random_seed=RANDOM_SEED, progressbar=False)
R_l, diag_l, div_l, flipped_l = align_and_extract(idata_l, "latent")
del idata_l, latent_model; gc.collect()
print(f"one-quality model: divergences = {div_l}, chains flipped = {flipped_l} of 4")
display(diag_l.round(3).head(8))

## How much of the A-level school effect does GCSE explain, with type in the model?

For each subject group, the true between-school variance splits into: **general GCSE quality** and the **tilt** (which now include the type and regional differences in those quantities), the **type shift beyond GCSE** (state types, plus the colleges and others), the **regional** part of the beyond-GCSE quality, the **shared A-level residual** $u_i$, and the group's **own scatter**. The table also shows the original "GCSE explains" figure without type.

In [ ]:
R = R_f
n = len(R["b"])
psi_all = np.concatenate([R["psi"], np.zeros((n, 1))], axis=1)
type_term = is4[None, None, :] * R["type_a"][:, :, t4_idx] + is_o[None, None, :] * R["d_o"][:, :, o_idx]      # (draws, group, schools)
comp = {"general GCSE quality": R["b"]**2 * R["g"].var(axis=1)[:, None], "tilt": R["c"]**2 * R["h"].var(axis=1)[:, None],
        "type beyond GCSE": type_term.var(axis=2), "regional (beyond GCSE)": R["lam_a"]**2 * psi_all[:, reg_idx].var(axis=1)[:, None],
        "shared A-level residual": R["lam_a"]**2 * R["u"].var(axis=1)[:, None], "group's own scatter": R["sd_group"]**2}
total = sum(comp.values())
shares = {k: v / total for k, v in comp.items()}
tab = pd.DataFrame({k: v.mean(axis=0) for k, v in shares.items()}, index=group_names)
tab["true SD"] = np.sqrt(total).mean(axis=0)
tab["GCSE explains (g + tilt)"] = (shares["general GCSE quality"] + shares["tilt"]).mean(axis=0)
tab["GCSE + type explain"] = (shares["general GCSE quality"] + shares["tilt"] + shares["type beyond GCSE"]).mean(axis=0)
tab["original: GCSE explains"] = orig["gcse_explains"]
display(tab.round(3))
del type_term; gc.collect()

fig, ax = plt.subplots(figsize=(9.5, 4.2))
left = np.zeros(n_groups)
colors = ["#4C72B0", "#55A868", "#937860", "#C44E52", "#8172B3", "#CCB974"]
for (k, _), col in zip(comp.items(), colors):
    v = shares[k].mean(axis=0)
    ax.barh(group_names, v, left=left, color=col, label=k); left += v
ax.invert_yaxis(); ax.set_xlabel("share of true between-school variance"); ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3, fontsize=8)
plt.show()

## The slopes on the two GCSE dimensions

$b_j$ (general GCSE quality) and $c_j$ (the tilt) per subject group, in A-level value-added points per SD, with 89% intervals from the model with type. The crosses are the original values without type.

In [ ]:
def forest(ax, draws, label, orig_vals, color):
    lo, med, hi = np.percentile(draws, [5.5, 50, 94.5], axis=0)
    for k in range(draws.shape[1]):
        ax.plot([lo[k], hi[k]], [k, k], color=color, linewidth=2.5); ax.plot(med[k], k, "o", color=color)
        ax.plot(orig_vals[k], k, "x", color="#C44E52", markersize=8, markeredgewidth=2, label="original, no type" if k == 0 else None)
    ax.axvline(0, color="grey", linewidth=0.8, linestyle="--"); ax.set_xlabel(label)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), sharey=True)
forest(axes[0], R_f["b"], "slope on general GCSE quality, $b_j$", orig["b"], "#4C72B0")
forest(axes[1], R_f["c"], "slope on the tilt (+: Maths/Science side), $c_j$", orig["c"], "#DD8452")
axes[0].set_yticks(range(n_groups), group_names); axes[0].invert_yaxis(); axes[1].legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()
pd.DataFrame({"b": R_f["b"].mean(axis=0), "b original": orig["b"], "c": R_f["c"].mean(axis=0), "c original": orig["c"],
              "shared-quality loading": R_f["lam_a"].mean(axis=0), "group scatter": R_f["sd_group"].mean(axis=0)}, index=group_names).round(3)

## Type at GCSE and at A-level

**At GCSE:** the shift in general quality (SD units), in the tilt (positive is the Maths-and-Science side) and in log consistency (positive is more scattered), for the four state types, relative to the average of the four. **At A-level:** the shift beyond the GCSE profile for each subject group (value-added points). The tables give the contrasts.

In [ ]:
colors4 = ["#4C72B0", "#DD8452", "#55A868", "#8172B3"]
tg, th, tc = R_f["type_g"], R_f["type_h"], R_f["type_c"]
lam_mean = R_f["lam"].mean()
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8), sharey=True)
for ax, arr, title in [(axes[0], tg, "general GCSE quality (SD units)"), (axes[1], th, "tilt (+ = Maths/Science side)"), (axes[2], tc, "log consistency (+ = more scattered)")]:
    lo, med, hi = np.percentile(arr, [5.5, 50, 94.5], axis=0)
    for k in range(4):
        ax.plot([lo[k], hi[k]], [k, k], color=colors4[k], linewidth=3); ax.plot(med[k], k, "o", color=colors4[k])
    ax.axvline(0, color="grey", linewidth=0.8, linestyle="--"); ax.set_xlabel(title)
axes[0].set_yticks(range(4), state4); axes[0].invert_yaxis()
plt.tight_layout(); plt.show()

pairs = [("academy converter", "academy sponsor-led"), ("academy converter", "LA maintained"), ("academy sponsor-led", "LA maintained"), ("free school / UTC / studio", "LA maintained")]
rows = []
for a, b_ in pairs:
    ia, ib = state4.index(a), state4.index(b_)
    for name, arr in [("general quality", tg), ("tilt", th), ("log consistency", tc)]:
        diff = arr[:, ia] - arr[:, ib]
        lo, hi = np.percentile(diff, [5.5, 94.5])
        rows.append({"contrast": f"{a} minus {b_}", "quantity": name, "difference": diff.mean(), "89% interval": f"[{lo:.3f}, {hi:.3f}]",
                     "P(> 0)": (diff > 0).mean(), "in GCSE points": diff.mean() * lam_mean if name == "general quality" else np.nan})
display(pd.DataFrame(rows).set_index(["contrast", "quantity"]).round(3))

ta = R_f["type_a"]
fig, ax = plt.subplots(figsize=(9, 5.2))
for k in range(4):
    lo, med, hi = np.percentile(ta[:, :, k], [5.5, 50, 94.5], axis=0)
    off = (k - 1.5) * 0.17
    for j in range(n_groups):
        ax.plot([lo[j], hi[j]], [j + off, j + off], color=colors4[k], linewidth=2.2, label=state4[k] if j == 0 else None)
        ax.plot(med[j], j + off, "o", color=colors4[k], markersize=4)
ax.axvline(0, color="grey", linewidth=0.8, linestyle="--")
ax.set_yticks(range(n_groups), group_names); ax.invert_yaxis()
ax.set_xlabel("shift in A-level value added beyond GCSE profile (points)"); ax.legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()
rows = []
for a, b_ in [("academy converter", "academy sponsor-led"), ("academy converter", "LA maintained"), ("academy sponsor-led", "LA maintained")]:
    ia, ib = state4.index(a), state4.index(b_)
    r = {"contrast": f"{a} minus {b_}"}
    for j, gname in enumerate(group_names):
        diff = ta[:, j, ia] - ta[:, j, ib]
        lo, hi = np.percentile(diff, [5.5, 94.5])
        r[gname] = f"{diff.mean():+.2f} [{lo:+.2f}, {hi:+.2f}]"
    rows.append(r)
display(pd.DataFrame(rows).set_index("contrast"))

## Regions, with and without type

The regional shift in general GCSE quality ($m_r$, SD units) and in the A-level quality beyond GCSE ($\psi_r$), from the model with type, next to the original values without type. If London's advantage were partly a matter of type mix (London has fewer converters and more maintained schools than other regions), it would shrink here; if type is separate from geography, it would not.

In [ ]:
m_d, psi_d = R_f["m"], R_f["psi"]
tab = pd.DataFrame({"m_r with type": m_d.mean(axis=0),
                    "89% interval": [f"[{np.percentile(m_d[:, k], 5.5):+.2f}, {np.percentile(m_d[:, k], 94.5):+.2f}]" for k in range(len(regions))],
                    "m_r original": [orig["m"][r] for r in regions],
                    "psi_r with type": psi_d.mean(axis=0),
                    "psi_r original": [orig["psi"][r] for r in regions],
                    "P(psi_r > 0)": (psi_d > 0).mean(axis=0)}, index=regions).sort_values("m_r with type", ascending=False)
display(tab.round(3))
order_r = np.argsort(-m_d.mean(axis=0))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), sharey=True)
for ax, arr, key, label, col in [(axes[0], m_d, "m", "general GCSE quality, $m_r$ (SD units)", "#4C72B0"), (axes[1], psi_d, "psi", "A-level shift beyond GCSE, $\\psi_r$", "#DD8452")]:
    lo, med, hi = np.percentile(arr, [5.5, 50, 94.5], axis=0)
    for row, k in enumerate(order_r):
        ax.plot([lo[k], hi[k]], [row, row], color=col, linewidth=2.5); ax.plot(med[k], row, "o", color=col)
        ax.plot(orig[key][regions[k]], row, "x", color="#C44E52", markersize=7, markeredgewidth=2, label="original, no type" if row == 0 else None)
    ax.axvline(0, color="grey", linewidth=0.8, linestyle="--"); ax.set_xlabel(label)
axes[0].set_yticks(range(len(regions)), [regions[k] for k in order_r]); axes[0].invert_yaxis(); axes[1].legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()

## One A-level quality, with type

The original model tied every group to a single A-level quality, $a_i = \beta_g g_i + \beta_h h_i + \psi_{r(i)} + \sigma_u u_i$, on which the groups load. Here it also has a **type shift** $q_{type}$ (in Maths-scale points; the four state types sum to zero). We report the same quantities as before: the correlation of A-level quality with general GCSE quality and with the tilt, and the share of its variance that GCSE explains, now also with the type shift; then how well the single quality reproduces the flexible slopes.

In [ ]:
Rl = R_l
nl = len(Rl["beta_g"])
psi_l = np.concatenate([Rl["psi"], np.zeros((nl, 1))], axis=1)
gc_part = Rl["beta_g"][:, None] * Rl["g"] + Rl["beta_h"][:, None] * Rl["h"]
type_part = is4[None, :] * Rl["type_q"][:, t4_idx]
a = gc_part + psi_l[:, reg_idx] + type_part + Rl["sigma_u"][:, None] * Rl["u"]
Va = a.var(axis=1)
corr_g = np.array([np.corrcoef(a[d], Rl["g"][d])[0, 1] for d in range(nl)])
corr_h = np.array([np.corrcoef(a[d], Rl["h"][d])[0, 1] for d in range(nl)])
expl_gcse = gc_part.var(axis=1) / Va
expl_all = (gc_part + type_part).var(axis=1) / Va
def s(x): return f"{x.mean():.3f} [{np.percentile(x, 5.5):.3f}, {np.percentile(x, 94.5):.3f}]"
print("beta_g:", s(Rl["beta_g"]), "(original 0.126) | beta_h:", s(Rl["beta_h"]), "(original 0.057) | sigma_u:", f"{Rl['sigma_u'].mean():.3f}", "(original 0.284)")
print("correlation of A-level quality with general GCSE quality:", s(corr_g), "(original 0.416) | with the tilt:", f"{corr_h.mean():.3f}", "(original 0.181)")
print("share of A-level quality variance explained by g and tilt:", s(expl_gcse), "(original 0.199) | by g, tilt and type beyond GCSE:", s(expl_all))
tq = Rl["type_q"]
print("type shifts in the one A-level quality (Maths-scale points):", {state4[k]: fmt for k, fmt in enumerate([s(tq[:, k]) for k in range(4)])})
imp_b, imp_c = Rl["lam_a"] * Rl["beta_g"][:, None], Rl["lam_a"] * Rl["beta_h"][:, None]
display(pd.DataFrame({"flexible b": R_f["b"].mean(axis=0), "implied b (loading x beta_g)": imp_b.mean(axis=0), "flexible c": R_f["c"].mean(axis=0), "implied c (loading x beta_h)": imp_c.mean(axis=0),
                      "loading, flexible": R_f["lam_a"].mean(axis=0), "loading, one-quality": Rl["lam_a"].mean(axis=0)}, index=group_names).round(3))
del a, gc_part, type_part; gc.collect()

In [ ]:
six = groups_per_school == n_groups
rows7 = six[ys_idx]
order7 = np.lexsort((yg_idx[rows7], ys_idx[rows7]))

def a_resid_corr(R, kind):
    sel = slice(0, None, 5)
    nu, sdg = R["nu"][sel], R["sd_group"][sel]
    g, h, u = R["g"][sel], R["h"][sel], R["u"][sel]
    nn = len(nu)
    psi = np.concatenate([R["psi"][sel], np.zeros((nn, 1))], axis=1)[:, reg_idx]
    lamA = R["lam_a"][sel]
    off = is_o[ys_idx][None, :] * R["d_o"][sel][:, yg_idx, o_idx[ys_idx]]
    if kind == "flex":
        typ = is4[ys_idx][None, :] * R["type_a"][sel][:, yg_idx, t4_idx[ys_idx]]
        mean = nu[:, yg_idx] + R["b"][sel][:, yg_idx] * g[:, ys_idx] + R["c"][sel][:, yg_idx] * h[:, ys_idx] + lamA[:, yg_idx] * (psi + u)[:, ys_idx] + typ + off
    else:
        q = R["beta_g"][sel][:, None] * g + R["beta_h"][sel][:, None] * h + psi + is4[None, :] * R["type_q"][sel][:, t4_idx] + R["sigma_u"][sel][:, None] * u
        mean = nu[:, yg_idx] + lamA[:, yg_idx] * q[:, ys_idx] + off
    sd = np.sqrt(sdg[:, yg_idx]**2 + y_se**2)
    out = {}
    for label, r in [("observed", (y_obs[None, :] - mean) / sd), ("replicated", rng.standard_normal(mean.shape))]:
        r7 = r[:, rows7][:, order7].reshape(r.shape[0], -1, n_groups)
        out[label] = np.stack([np.corrcoef(r7[d].T) for d in range(r7.shape[0])])
    return out

res_f, res_l = a_resid_corr(R_f, "flex"), a_resid_corr(R_l, "latent")
fig, axes = plt.subplots(1, 3, figsize=(15, 5), gridspec_kw={"width_ratios": [1, 1, 0.04]})
for ax, res, title in [(axes[0], res_f, "flexible, with type"), (axes[1], res_l, "one A-level quality, with type")]:
    corr = res["observed"].mean(axis=0)
    im = ax.imshow(corr, vmin=-0.3, vmax=0.3, cmap="RdBu_r")
    ax.set_xticks(range(n_groups), group_names, rotation=35, ha="right"); ax.set_yticks(range(n_groups), group_names)
    for i in range(n_groups):
        for j in range(n_groups):
            ax.text(j, i, f"{corr[i, j]:.2f}", ha="center", va="center", fontsize=8)
    ax.set_title(f"A-level residual correlation, {title}"); ax.grid(False)
fig.colorbar(im, cax=axes[2])
plt.show()
off_diag = ~np.eye(n_groups, dtype=bool)
print(f"{six.sum()} schools with all {n_groups} groups; null (replicated) 89% upper bound for a single pair is about {np.percentile(res_f['replicated'][:, 0, 1], 94.5):.3f}")
print("largest off-diagonal residual correlation: flexible", np.abs(res_f['observed'].mean(axis=0)[off_diag]).max().round(3), "| one A-level quality", np.abs(res_l['observed'].mean(axis=0)[off_diag]).max().round(3), "(originally 0.097 and 0.227)")

## Summary: what including type changes

Both fits have no divergences. The weakest parameters are the overall GCSE level $\mu_e$ ($\hat R$ 1.11 to 1.12, ESS 26 to 27) and the type effects on general GCSE quality (ESS 70 to 86); everything else mixes better (ESS above 140). All numbers are from 500 draws per chain, and the original values are those stored in the cell near the top.

**The original conclusions stand, and type refines them.**

- **GCSE plus type explains a little more than GCSE alone did.** Adding the type shift beyond GCSE: Maths 0.33 (original 0.28), Sciences 0.33 (0.28), Business & Computing 0.11 (0.08), Creative arts 0.055 (0.042); English 0.18 (0.17), Humanities 0.20 (0.19) and Social sciences 0.12 (0.12) barely change. Type beyond the GCSE profile accounts for 2% (Social sciences) to 6% (Business & Computing, Sciences) of the true school-to-school variance.
- **Part of what the original credited to general GCSE quality was type.** With type in the model the slope on general GCSE quality is 10% to 23% lower in every group (Maths 0.080 against 0.100, Sciences 0.070 against 0.088, Business & Computing 0.060 against 0.078), and the share of variance carried by general quality falls (Maths 4.8% against 7.1%), because converters are both higher at GCSE and higher at A-level.
- **The tilt link is unaffected.** The slopes on the tilt are almost unchanged: Maths $+0.19$ (original $+0.18$), Sciences $+0.15$ ($+0.15$), and the small negative ones for English, Humanities and Social sciences. The Maths-and-Science persistence between GCSE and A-level is not a type effect.

**Type itself** (all effects as in `a-level-institution-type.ipynb`, on this smaller sample without independent schools and without the local-authority layer):

- At GCSE, converters are above sponsor-led academies in general quality by $0.68$ SD [0.58, 0.78] and more consistent; against LA-maintained schools converters are $+0.35$ [0.24, 0.45] and sponsor-led $-0.33$ [-0.46, -0.21]; free schools, UTCs and studio schools lean to the Maths-and-Science side ($+0.80$ [0.53, 1.06]).
- At A-level beyond the GCSE profile, converters beat sponsor-led academies in every group (Social sciences $+0.05$ to Business & Computing $+0.15$ points), look like maintained schools except in Sciences ($+0.06$ [0.02, 0.09]), and sponsor-led academies sit below maintained schools, clearly so in Maths ($-0.11$), Sciences ($-0.06$), Humanities ($-0.07$), Social sciences ($-0.05$) and Business & Computing ($-0.14$).

**Regions.** London's GCSE advantage is not a matter of type mix: with type it is $+0.70$ SD [0.61, 0.79], slightly larger than the original $+0.62$, because London has fewer converters. The other regions are essentially unchanged (South East $+0.13$, North East $-0.37$). The regional shifts in A-level quality beyond GCSE keep their pattern: East of England $+0.17$, London $+0.12$ (original $+0.07$), West Midlands $-0.14$, North East $-0.16$; small in value-added points once multiplied by the groups' loadings.

**One A-level quality.** With type the link between A-level quality and general GCSE quality is a correlation of 0.40 (original 0.42), and with the tilt 0.19 (0.18). GCSE explains 17% of the quality's variance (original 20%) and GCSE plus the type shift 21% [17%, 26%]. The type shifts in the quality are converters $+0.065$ [0.041, 0.088], LA maintained $+0.045$, sponsor-led $-0.052$ [-0.081, -0.022] and free schools/UTCs $-0.058$ (Maths-scale points). But the single quality still cannot represent how GCSE reaches the groups: the largest cross-group residual correlation is 0.22 against 0.10 for the flexible model, as originally (0.23 and 0.10). The groups still need their own slopes.

**Caveats.** As in the original, GCSE describes this year's Year 11 and A-level students took their GCSEs two years earlier, and pooled standard errors assume separate cohorts. Type effects reflect who converted and where sponsor-led academies were created, not the effect of type. The 19 colleges and other institutions that have GCSE results are given an offset and are not interpreted.